In [ ]:
import pandas as pd

# E-commerce Fraud Detection Analysis: VPN Transactions Deep Dive

## Introduction

This notebook analyzes an e-commerce fraud dataset to understand the impact of VPN-detected transactions. Out of 10,000 transactions, 619 were flagged for VPN usage, indicating potential location spoofing or suspicious activity. This analysis focuses on identifying patterns in categories, browsers, and card types associated with VPN transactions to inform fraud detection strategies.

### Objectives
- Identify the most impacted product categories by VPN transactions.
- Determine the most commonly used browsers in VPN transactions.
- Analyze card type preferences in VPN transactions.
- Provide insights and recommendations for fraud prevention.

### Dataset Overview
The dataset contains transaction data including amounts, categories, payment methods, device information, and fraud indicators.

In [29]:
df = pd.read_csv('ecommerce_fraud_dataset.csv')
df.head()

,transaction_id,transaction_date,hour,amount_usd,category,country,payment_method,card_type,device_type,browser,account_age_days,tx_last_24h,n_items,addr_mismatch,form_fill_time_sec,email_domain,vpn_detected,is_fraud
0,TXN1006252,2023-03-20 08:59:00,8,86.81,Fashion,FR,credit_card,Visa,desktop,Firefox,477,1,3,1,92,outlook.com,0,0
1,TXN1004684,2023-12-26 10:14:00,10,1117.30,Travel,FR,digital_wallet,Discover,desktop,Chrome,573,1,4,0,119,hotmail.com,0,0
2,TXN1001731,2023-04-21 14:37:00,14,496.56,Luxury,UK,debit_card,Visa,tablet,Firefox,674,1,1,0,93,yahoo.com,0,0
3,TXN1004742,2023-12-21 13:13:00,13,163.55,Fashion,UK,digital_wallet,Mastercard,desktop,Safari,653,4,3,0,47,icloud.com,0,0
4,TXN1004521,2023-11-15 08:56:00,8,562.20,Travel,US,digital_wallet,Mastercard,desktop,Unknown,305,1,1,0,55,outlook.com,0,0


## Data Overview

Let's examine the dataset structure and basic statistics.

In [37]:
# Basic dataset information
print("Dataset shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nBasic statistics:")
print(df.describe())

# VPN detection summary
vpn_percentage = (df['vpn_detected'].sum() / len(df)) * 100
print(f"\nVPN Detection Rate: {vpn_percentage:.2f}% ({df['vpn_detected'].sum()} out of {len(df)} transactions)")

# Fraud rate
fraud_percentage = (df['is_fraud'].sum() / len(df)) * 100
print(f"Fraud Rate: {fraud_percentage:.2f}% ({df['is_fraud'].sum()} out of {len(df)} transactions)")

Dataset shape: (10000, 18)

Columns: ['transaction_id', 'transaction_date', 'hour', 'amount_usd', 'category', 'country', 'payment_method', 'card_type', 'device_type', 'browser', 'account_age_days', 'tx_last_24h', 'n_items', 'addr_mismatch', 'form_fill_time_sec', 'email_domain', 'vpn_detected', 'is_fraud']

Data types:
transaction_id            str
transaction_date          str
hour                    int64
amount_usd            float64
category                  str
country                   str
payment_method            str
card_type                 str
device_type               str
browser                   str
account_age_days        int64
tx_last_24h             int64
n_items                 int64
addr_mismatch           int64
form_fill_time_sec      int64
email_domain              str
vpn_detected            int64
is_fraud                int64
dtype: object

Missing values:
transaction_id        0
transaction_date      0
hour                  0
amount_usd            0
category     

In [10]:
df[df['vpn_detected']>0]

,transaction_id,transaction_date,hour,amount_usd,category,country,payment_method,card_type,device_type,browser,account_age_days,tx_last_24h,n_items,addr_mismatch,form_fill_time_sec,email_domain,vpn_detected,is_fraud
5,TXN1006340,2023-07-11 16:52:00,16,133.55,Fashion,BR,credit_card,Amex,mobile,Chrome,1674,2,2,0,138,hotmail.com,1,0
17,TXN1009930,2023-04-10 09:05:00,9,260.63,Electronics,BR,digital_wallet,Mastercard,desktop,Safari,1841,2,1,0,12,outlook.com,1,0
37,TXN1000251,2023-07-13 21:24:00,21,400.13,Luxury,NG,digital_wallet,Visa,mobile,Chrome,125,4,1,1,6,icloud.com,1,1
45,TXN1000039,2023-06-07 11:23:00,11,240.69,Gift Cards,FR,credit_card,Visa,desktop,Chrome,5,6,4,0,2,hotmail.com,1,1
73,TXN1002894,2023-07-21 06:07:00,6,1009.79,Luxury,IN,credit_card,Visa,desktop,Chrome,1894,1,1,0,123,outlook.com,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9903,TXN1005892,2023-03-26 14:44:00,14,202.38,Travel,DE,credit_card,Visa,desktop,Chrome,419,3,3,0,90,yahoo.com,1,0
9914,TXN1000064,2023-04-19 02:29:00,2,64.96,Software,DE,credit_card,Discover,mobile,Edge,0,9,5,1,14,outlook.com,1,1
9931,TXN1001016,2023-01-06 16:43:00,16,303.30,Electronics,IN,credit_card,Mastercard,mobile,Chrome,1028,3,3,1,185,gmail.com,1,0
9933,TXN1007629,2023-12-19 13:22:00,13,430.52,Travel,US,credit_card,Visa,tablet,Chrome,1410,1,1,0,2,gmail.com,1,0


Out of 10,000 transactions, 619 use a VPN. This means these transactions are potentially relocated or the declared location is not reliable. We will analyze in detail:

- which category is most affected by VPN transactions;
- which browser is most used;
- which card type is most used (Mastercard, Visa, etc.).

In [34]:
# VPN transaction analysis

df_vpn = df[df['vpn_detected'] > 0].copy()
vpn_total = len(df_vpn)

category_counts = df_vpn['category'].value_counts()
category_share = (category_counts / df['category'].value_counts() * 100).round(2)

browser_counts = df_vpn['browser'].value_counts()
card_counts = df_vpn['card_type'].value_counts()

most_impacted_category = category_counts.idxmax()
most_impacted_category_count = int(category_counts.iloc[0])
most_impacted_category_share = float(category_share.loc[most_impacted_category])

most_used_browser = browser_counts.idxmax()
most_used_browser_count = int(browser_counts.iloc[0])

most_used_card_type = card_counts.idxmax()
most_used_card_type_count = int(card_counts.iloc[0])

print(f"Total transactions with VPN: {vpn_total} / {len(df)}")
print()
print("Top 5 categories most affected by VPN:")
print(category_counts.head(5))
print()
print("Share of VPN in these categories (percentage of each category total):")
print(category_share.loc[category_counts.head(5).index])
print()
print("Top 5 browsers used in VPN transactions:")
print(browser_counts.head(5))
print()
print("Top 5 card types used in VPN transactions:")
print(card_counts.head(5))

Total transactions with VPN: 619 / 10000

Top 5 categories most affected by VPN:
category
Electronics    130
Fashion        115
Travel          85
Gift Cards      80
Gaming          66
Name: count, dtype: int64

Share of VPN in these categories (percentage of each category total):
category
Electronics    6.17
Fashion        4.72
Travel         5.83
Gift Cards     9.77
Gaming         5.66
Name: count, dtype: float64

Top 5 browsers used in VPN transactions:
browser
Chrome     303
Safari     114
Unknown     89
Edge        66
Firefox     47
Name: count, dtype: int64

Top 5 card types used in VPN transactions:
card_type
Discover      170
Visa          157
Amex          155
Mastercard    137
Name: count, dtype: int64


### Summary of results

- Category most impacted by VPN transactions: **{most_impacted_category}** ({most_impacted_category_count} VPN transactions, representing {most_impacted_category_share} % of that category).
- Browser most used in VPN transactions: **{most_used_browser}** ({most_used_browser_count} transactions).
- Card type most used in VPN transactions: **{most_used_card_type}** ({most_used_card_type_count} transactions).

This analysis focuses only on transactions where `vpn_detected > 0`, to target potentially relocated or suspicious segments.

## Comparative Analysis

Let's compare VPN transactions with non-VPN transactions to identify patterns.

In [36]:
# Compare category distributions
df_no_vpn = df[df['vpn_detected'] == 0]
category_no_vpn = df_no_vpn['category'].value_counts()

# Create comparison dataframe
comparison_df = pd.DataFrame({
    'VPN': category_counts,
    'No VPN': category_no_vpn
}).fillna(0)

comparison_df['VPN_Percentage'] = (comparison_df['VPN'] / comparison_df['VPN'].sum()) * 100
comparison_df['No_VPN_Percentage'] = (comparison_df['No VPN'] / comparison_df['No VPN'].sum()) * 100

print("Category comparison (VPN vs No VPN):")
print(comparison_df.head(10))

# Fraud correlation with VPN
fraud_vpn = df_vpn['is_fraud'].sum()
fraud_no_vpn = df_no_vpn['is_fraud'].sum()

print(f"\nFraud in VPN transactions: {fraud_vpn} ({fraud_vpn/len(df_vpn)*100:.2f}%)")
print(f"Fraud in non-VPN transactions: {fraud_no_vpn} ({fraud_no_vpn/len(df_no_vpn)*100:.2f}%)")

# Average transaction amounts
avg_amount_vpn = df_vpn['amount_usd'].mean()
avg_amount_no_vpn = df_no_vpn['amount_usd'].mean()

print(f"\nAverage transaction amount - VPN: ${avg_amount_vpn:.2f}")
print(f"Average transaction amount - No VPN: ${avg_amount_no_vpn:.2f}")

Category comparison (VPN vs No VPN):
             VPN  No VPN  VPN_Percentage  No_VPN_Percentage
category                                                   
Electronics  130    1976       21.001616          21.063852
Fashion      115    2322       18.578352          24.752159
Gaming        66    1101       10.662359          11.736489
Gift Cards    80     739       12.924071           7.877625
Jewelry       40     471        6.462036           5.020787
Luxury        53     663        8.562197           7.067477
Software      50     735        8.077544           7.834986
Travel        85    1374       13.731826          14.646626

Fraud in VPN transactions: 229 (37.00%)
Fraud in non-VPN transactions: 131 (1.40%)

Average transaction amount - VPN: $273.96
Average transaction amount - No VPN: $266.13


In [35]:
print(f"Summary: {vpn_total} VPN transactions detected out of {len(df)} total transactions.")
print(f"Most impacted category: {most_impacted_category} ({most_impacted_category_count} transactions, {most_impacted_category_share} % of the category).")
print(f"Most used browser: {most_used_browser} ({most_used_browser_count} VPN transactions).")
print(f"Most used card type: {most_used_card_type} ({most_used_card_type_count} VPN transactions).")

Summary: 619 VPN transactions detected out of 10000 total transactions.
Most impacted category: Electronics (130 transactions, 6.17 % of the category).
Most used browser: Chrome (303 VPN transactions).
Most used card type: Discover (170 VPN transactions).


## Conclusions and Recommendations

### Key Findings
1. **VPN Usage Rate**: 6.19% of transactions show VPN detection, potentially indicating location spoofing.
2. **Category Impact**: [Most impacted category] shows the highest concentration of VPN transactions.
3. **Browser Preferences**: [Most used browser] is predominantly used in VPN transactions.
4. **Card Type Patterns**: [Most used card type] is the most common in VPN transactions.
5. **Fraud Correlation**: VPN transactions show a higher fraud rate compared to non-VPN transactions.

### Business Insights
- VPN-detected transactions may represent higher risk segments.
- Certain categories and browsers are more associated with suspicious activity.
- Monitoring VPN usage could be a key fraud prevention strategy.

### Recommendations
1. **Enhanced Monitoring**: Implement stricter checks for transactions involving VPNs, especially in high-risk categories.
2. **Browser-Based Rules**: Consider browser-specific fraud detection rules.
3. **Geolocation Verification**: Develop additional verification methods beyond IP-based location detection.
4. **Customer Education**: Inform customers about security best practices.
5. **Further Analysis**: Investigate correlations with other variables like transaction amounts and device types.

This analysis provides a foundation for understanding VPN-related fraud patterns. Further statistical modeling and machine learning could enhance detection capabilities.